### Cell 1 — Setup

In [0]:
# ============================================================
#  04c_ANOMALY_DETECTION — TOURGO ML PIPELINE
#  Day 8: Phát hiện bất thường bằng Z-score / thống kê
#  Input : silver_bookings, silver_revenues, gold_revenue_daily
#  Output: gold_anomalies
# ============================================================

from pyspark.sql.functions import (
    col, avg, stddev, count, sum as _sum,
    when, lit, to_date, abs as _abs,
    round as _round
)

spark.sql("USE tourgo")

df_bookings = spark.read.table("silver_bookings")
df_revenues  = spark.read.table("silver_revenues")
df_rev_daily = spark.read.table("gold_revenue_daily")

print("=> Loaded:")
print(f"   bookings : {df_bookings.count():,}")
print(f"   revenues : {df_revenues.count():,}")
print(f"   rev_daily: {df_rev_daily.count():,} ngày")

=> Loaded:
   bookings : 358
   revenues : 114
   rev_daily: 32 ngày


### Cell 2 — Anomaly Type 1: User đặt quá nhiều

In [0]:
# ── TYPE 1: User booking bất thường trong 1 ngày ───────────
print("=>  Tính Anomaly Type 1: User booking bất thường...")

df_daily_user = df_bookings \
    .withColumn("date", to_date(col("created_at"))) \
    .groupBy("user_id", "date") \
    .agg(count("id").alias("bookings_per_day"))

stats_u = df_daily_user.agg(
    avg("bookings_per_day").alias("mean"),
    stddev("bookings_per_day").alias("std")
).collect()[0]

mean_u = stats_u["mean"] or 0
std_u  = stats_u["std"]  or 0

# Dùng 2-sigma thay vì 3-sigma nếu data ít
SIGMA = 2 if df_daily_user.count() < 50 else 3
threshold_user = mean_u + SIGMA * std_u

df_anomaly_type1 = df_daily_user \
    .filter(col("bookings_per_day") > threshold_user) \
    .withColumn("anomaly_type",
                lit("User booking bất thường trong 1 ngày")) \
    .withColumn("severity", lit("HIGH")) \
    .withColumn("value", col("bookings_per_day").cast("double")) \
    .select(
        col("user_id"),
        lit(None).cast("integer").alias("tour_id"),
        col("date"),
        "anomaly_type", "severity", "value"
    )

count1 = df_anomaly_type1.count()
print(f"   Mean: {mean_u:.2f} | Std: {std_u:.2f} | "
      f"Threshold ({SIGMA}σ): {threshold_user:.2f}")
print(f"   -> {count1} anomalies phát hiện")
df_anomaly_type1.show(5, truncate=False)

=>  Tính Anomaly Type 1: User booking bất thường...
   Mean: 1.01 | Std: 0.07 | Threshold (3σ): 1.23
   -> 2 anomalies phát hiện
+-------+-------+----------+------------------------------------+--------+-----+
|user_id|tour_id|date      |anomaly_type                        |severity|value|
+-------+-------+----------+------------------------------------+--------+-----+
|328    |NULL   |2026-04-25|User booking bất thường trong 1 ngày|HIGH    |2.0  |
|340    |NULL   |2026-05-10|User booking bất thường trong 1 ngày|HIGH    |2.0  |
+-------+-------+----------+------------------------------------+--------+-----+



### Cell 3 — Anomaly Type 2: Tour cancellation rate cao

In [0]:
# ── TYPE 2: Tour cancellation rate bất thường ──────────────
print("=>  Tính Anomaly Type 2: Tour cancellation bất thường...")

from pyspark.sql.functions import upper as _upper

df_tour_cancel = df_bookings \
    .groupBy("tour_id") \
    .agg(
        count("id").alias("total"),
        count(when(_upper(col("status")) == "CANCELLED", 1))
            .alias("cancelled")
    ) \
    .withColumn("cancel_rate", col("cancelled") / col("total")) \
    .filter(col("total") >= 3)  # ít nhất 3 bookings

stats_c = df_tour_cancel.agg(
    avg("cancel_rate").alias("mean"),
    stddev("cancel_rate").alias("std")
).collect()[0]

mean_c = stats_c["mean"] or 0
std_c  = stats_c["std"]  or 0
threshold_cancel = mean_c + 2 * std_c

df_anomaly_type2 = df_tour_cancel \
    .filter(col("cancel_rate") > threshold_cancel) \
    .withColumn("anomaly_type",
                lit("Tour có tỷ lệ hủy bất thường")) \
    .withColumn("severity",
                when(col("cancel_rate") > 0.7, "HIGH")
                .otherwise("MEDIUM")) \
    .withColumn("value", col("cancel_rate")) \
    .select(
        lit(None).cast("integer").alias("user_id"),
        col("tour_id"),
        lit(None).cast("date").alias("date"),
        "anomaly_type", "severity", "value"
    )

count2 = df_anomaly_type2.count()
print(f"   Mean: {mean_c:.3f} | Threshold: {threshold_cancel:.3f}")
print(f"   -> {count2} tours bất thường")
df_anomaly_type2.show(5, truncate=False)

=>  Tính Anomaly Type 2: Tour cancellation bất thường...
   Mean: 0.064 | Threshold: 0.313
   -> 8 tours bất thường
+-------+-------+----+----------------------------+--------+------------------+
|user_id|tour_id|date|anomaly_type                |severity|value             |
+-------+-------+----+----------------------------+--------+------------------+
|NULL   |48     |NULL|Tour có tỷ lệ hủy bất thường|MEDIUM  |0.3333333333333333|
|NULL   |39     |NULL|Tour có tỷ lệ hủy bất thường|MEDIUM  |0.3333333333333333|
|NULL   |15     |NULL|Tour có tỷ lệ hủy bất thường|MEDIUM  |0.3333333333333333|
|NULL   |2      |NULL|Tour có tỷ lệ hủy bất thường|MEDIUM  |0.3333333333333333|
|NULL   |89     |NULL|Tour có tỷ lệ hủy bất thường|MEDIUM  |0.3333333333333333|
+-------+-------+----+----------------------------+--------+------------------+
only showing top 5 rows


### Cell 4 — Anomaly Type 3: Revenue spike/drop

In [0]:
# ── TYPE 3: Revenue spike hoặc drop theo ngày ──────────────
print("=>  Tính Anomaly Type 3: Revenue spike/drop...")

stats_r = df_rev_daily.agg(
    avg("total_revenue").alias("mean"),
    stddev("total_revenue").alias("std")
).collect()[0]

mean_r = stats_r["mean"] or 0
std_r  = stats_r["std"]  or 1  # tránh chia 0

df_anomaly_type3 = df_rev_daily \
    .withColumn("z_score",
                (col("total_revenue") - mean_r) / std_r) \
    .filter((_abs(col("z_score")) > 2.0)) \
    .withColumn("anomaly_type",
                when(col("z_score") > 2.0, "Revenue đột biến tăng")
                .otherwise("Revenue đột biến giảm")) \
    .withColumn("severity",
                when(_abs(col("z_score")) > 3.0, "HIGH")
                .otherwise("MEDIUM")) \
    .withColumn("value", col("total_revenue")) \
    .select(
        lit(None).cast("integer").alias("user_id"),
        lit(None).cast("integer").alias("tour_id"),
        col("booking_date").alias("date"),
        "anomaly_type", "severity", "value"
    )

count3 = df_anomaly_type3.count()
print(f"   Mean: {mean_r:,.0f} | Std: {std_r:,.0f}")
print(f"   -> {count3} ngày bất thường")
df_anomaly_type3.show(5, truncate=False)

=>  Tính Anomaly Type 3: Revenue spike/drop...
   Mean: 20,343,750 | Std: 12,797,680
   -> 1 ngày bất thường
+-------+-------+----------+---------------------+--------+------+
|user_id|tour_id|date      |anomaly_type         |severity|value |
+-------+-------+----------+---------------------+--------+------+
|NULL   |NULL   |2026-05-09|Revenue đột biến tăng|MEDIUM  |4.93E7|
+-------+-------+----------+---------------------+--------+------+



### Cell 5 — Union & Lưu Gold Table

In [0]:
# ── Union 3 loại anomaly → Gold table ──────────────────────
from functools import reduce
from pyspark.sql import DataFrame

dfs = [df_anomaly_type1, df_anomaly_type2, df_anomaly_type3]
df_all = reduce(DataFrame.union, dfs)

# Nếu không có anomaly nào → tạo record demo để pipeline không rỗng
if df_all.count() == 0:
    print("--WARNING--  Không phát hiện anomaly — tạo record demo...")
    from datetime import date
    demo_data = [(None, None, str(date.today()),
                  "Demo: Không có dữ liệu bất thường thật",
                  "LOW", 0.0)]
    demo_schema = ["user_id","tour_id","date",
                   "anomaly_type","severity","value"]
    df_all = spark.createDataFrame(demo_data, demo_schema)

df_all.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_anomalies")

total_anomalies = spark.read.table("gold_anomalies").count()
print(f"\n=> gold_anomalies: {total_anomalies} records")
print(f"   Type 1 (user):    {count1}")
print(f"   Type 2 (tour):    {count2}")
print(f"   Type 3 (revenue): {count3}")

spark.read.table("gold_anomalies") \
    .orderBy(col("severity").desc(), "date") \
    .show(truncate=False)


=> gold_anomalies: 11 records
   Type 1 (user):    2
   Type 2 (tour):    8
   Type 3 (revenue): 1
+-------+-------+----------+------------------------------------+--------+------------------+
|user_id|tour_id|date      |anomaly_type                        |severity|value             |
+-------+-------+----------+------------------------------------+--------+------------------+
|NULL   |228    |NULL      |Tour có tỷ lệ hủy bất thường        |MEDIUM  |0.3333333333333333|
|NULL   |159    |NULL      |Tour có tỷ lệ hủy bất thường        |MEDIUM  |0.3333333333333333|
|NULL   |129    |NULL      |Tour có tỷ lệ hủy bất thường        |MEDIUM  |0.3333333333333333|
|NULL   |89     |NULL      |Tour có tỷ lệ hủy bất thường        |MEDIUM  |0.3333333333333333|
|NULL   |15     |NULL      |Tour có tỷ lệ hủy bất thường        |MEDIUM  |0.3333333333333333|
|NULL   |39     |NULL      |Tour có tỷ lệ hủy bất thường        |MEDIUM  |0.3333333333333333|
|NULL   |2      |NULL      |Tour có tỷ lệ hủy bất thườ

### Cell 6 — Checklist Day 7 & 8

In [0]:
# ============================================================
#  CHECKLIST NGÀY 7 + 8
# ============================================================

spark.sql("USE tourgo")

print("=" * 60)
print("=> CHECKLIST NGÀY 7 + 8")
print("=" * 60)

all_tables = {
    "Day 7": [
        "ml_revenue_forecast",
        "ml_tour_recommendations",
    ],
    "Day 8": [
        "gold_anomalies",
    ]
}

for day, tables in all_tables.items():
    print(f"\n  [{day}]")
    for t in tables:
        try:
            n = spark.read.table(t).count()
            print(f"    {t:<35} | --OK-- {n:,} records")
        except:
            print(f"    {t:<35} | --ERROR Chưa có")

# MLflow check
print("\n  [MLflow]")
try:
    runs = mlflow.search_runs(
        experiment_names=["tourgo-ml-experiments"]
    )
    print(f"    Tổng số runs logged: {len(runs)}")
    for _, run in runs.iterrows():
        name = run.get("tags.mlflow.runName", "unknown")
        r2   = run.get("metrics.r2", "—")
        sil  = run.get("metrics.final_silhouette", "—")
        print(f"    {name:<35} | r2={r2}  silhouette={sil}")
except Exception as e:
    print(f"    --WARNING--  {e}")

print("\n" + "=" * 60)
print("=> Checklist thủ công:")
print("  --> [E] Tân: chạy full pipeline test, ghi thời gian")
print("  --> [C] Phụng: Dashboard Section 6 (forecast chart)")
print(" --> [C] Phụng: Dashboard Section 8 (anomaly table)")
print("  --> Chụp screenshot MLflow: Experiments → 2 experiments")
print(" --> [D]+[E]: commit tất cả notebooks lên GitHub")

=> CHECKLIST NGÀY 7 + 8

  [Day 7]
    ml_revenue_forecast                 | --OK-- 7 records
    ml_tour_recommendations             | --OK-- 23 records

  [Day 8]
    gold_anomalies                      | --OK-- 11 records

  [MLflow]
    --WARNING--  name 'mlflow' is not defined

=> Checklist thủ công:
  --> [E] Tân: chạy full pipeline test, ghi thời gian
  --> [C] Phụng: Dashboard Section 6 (forecast chart)
 --> [C] Phụng: Dashboard Section 8 (anomaly table)
  --> Chụp screenshot MLflow: Experiments → 2 experiments
 --> [D]+[E]: commit tất cả notebooks lên GitHub
